# 01 - Dataset Inventory

## Objetivo

Realizar un inventario automático del dataset **Home Credit Default Risk** para conocer la estructura de las fuentes de datos antes del diseño del modelo relacional en PostgreSQL.

## Objetivos específicos

- Identificar todos los archivos disponibles.
- Obtener dimensiones de cada dataset.
- Detectar posibles claves (`SK_ID_*`).
- Identificar el rol de cada tabla dentro del modelo de datos.
- Generar un inventario reproducible que sirva como base para el diseño del esquema relacional.

---

In [117]:
# ==========================================================
# Imports
# ==========================================================

from pathlib import Path

import pandas as pd

In [118]:
# ==========================================================
# Project paths
# ==========================================================

PROJECT_ROOT = Path.cwd().parents[1]

DATA_PATH = PROJECT_ROOT / "data"

RAW_DATA_PATH = DATA_PATH / "raw"
PROCESSED_DATA_PATH = DATA_PATH / "processed"
ANALYTICAL_DATA_PATH = DATA_PATH / "analytical"

DOCS_PATH = PROJECT_ROOT / "docs"

In [119]:
# ==========================================================
# Dataset discovery
# ==========================================================

csv_files = sorted(RAW_DATA_PATH.glob("*.csv"))

print(f"CSV files found: {len(csv_files)}\n")

for file in csv_files:
    print(file.name)

CSV files found: 10

application_test.csv
application_train.csv
bureau.csv
bureau_balance.csv
credit_card_balance.csv
HomeCredit_columns_description.csv
installments_payments.csv
POS_CASH_balance.csv
previous_application.csv
sample_submission.csv


In [120]:
# ==========================================================
# Helper functions
# ==========================================================

def get_file_size_mb(file_path: Path) -> float:
    """
    Calculate the size of a file in megabytes.

    Parameters
    ----------
    file_path : Path
        Path to the file.

    Returns
    -------
    float
        File size in MB rounded to two decimals.
    """
    return round(file_path.stat().st_size / (1024 * 1024), 2)

In [121]:
# ==========================================================
# Dataset metadata
# ==========================================================

def get_dataset_metadata(file_path: Path) -> dict:
    """
    Extract basic metadata from a CSV dataset.

    Parameters
    ----------
    file_path : Path
        Path to the CSV file.

    Returns
    -------
    dict
        Dictionary containing the dataset metadata.
    """

    
    df = pd.read_csv(file_path, nrows=5, encoding="latin1")
    candidate_keys = [
        column
        for column in df.columns
        if column.startswith("SK_ID_")
    ]

    total_rows = sum(1 for _ in open(file_path, encoding="latin1")) - 1

    return {
        "file_name": file_path.name,
        "rows": total_rows,
        "columns": len(df.columns),
        "size_mb": get_file_size_mb(file_path),
        "candidate_keys": ", ".join(candidate_keys)
        if candidate_keys
        else "-"
    }

In [122]:
# ==========================================================
# Function validation
# ==========================================================

get_dataset_metadata(RAW_DATA_PATH / "application_train.csv")

{'file_name': 'application_train.csv',
 'rows': 307511,
 'columns': 122,
 'size_mb': 158.44,
 'candidate_keys': 'SK_ID_CURR'}

In [123]:
# ==========================================================
# Dataset inventory
# ==========================================================

inventory = [
    get_dataset_metadata(file)
    for file in csv_files
]

inventory_df = pd.DataFrame(inventory)

inventory_df

,file_name,rows,columns,size_mb,candidate_keys
0,application_test.csv,48744,121,25.34,SK_ID_CURR
1,application_train.csv,307511,122,158.44,SK_ID_CURR
2,bureau.csv,1716428,17,162.14,"SK_ID_CURR, SK_ID_BUREAU"
3,bureau_balance.csv,27299925,3,358.19,SK_ID_BUREAU
4,credit_card_balance.csv,3840312,23,404.91,"SK_ID_PREV, SK_ID_CURR"
5,HomeCredit_columns_description.csv,219,5,0.04,-
6,installments_payments.csv,13605401,8,689.62,"SK_ID_PREV, SK_ID_CURR"
7,POS_CASH_balance.csv,10001358,8,374.51,"SK_ID_PREV, SK_ID_CURR"
8,previous_application.csv,1670214,37,386.21,"SK_ID_PREV, SK_ID_CURR"
9,sample_submission.csv,48744,2,0.51,SK_ID_CURR


In [124]:
ENTITY_TYPES = {
    "application_train.csv": "Main application - train",
    "application_test.csv": "Main application - test",
    "bureau.csv": "Credit bureau history",
    "bureau_balance.csv": "Credit bureau monthly status",
    "previous_application.csv": "Previous credit applications",
    "POS_CASH_balance.csv": "POS cash monthly balance",
    "credit_card_balance.csv": "Credit card monthly balance",
    "installments_payments.csv": "Installment payments history",
    "sample_submission.csv": "Kaggle submission template",
    "HomeCredit_columns_description.csv": "Data dictionary"
}

PRIMARY_KEYS = {
    "application_train.csv": "SK_ID_CURR",
    "application_test.csv": "SK_ID_CURR",
    "bureau.csv": "SK_ID_BUREAU",
    "bureau_balance.csv": "SK_ID_BUREAU, MONTHS_BALANCE",
    "previous_application.csv": "SK_ID_PREV",
    "POS_CASH_balance.csv": "SK_ID_PREV, MONTHS_BALANCE",
    "credit_card_balance.csv": "SK_ID_PREV, MONTHS_BALANCE",
    "installments_payments.csv": "SK_ID_PREV, NUM_INSTALMENT_NUMBER, NUM_INSTALMENT_VERSION",
    "sample_submission.csv": "SK_ID_CURR",
    "HomeCredit_columns_description.csv": "-"
}

FOREIGN_KEYS = {
    "application_train.csv": "-",
    "application_test.csv": "-",
    "bureau.csv": "SK_ID_CURR",
    "bureau_balance.csv": "SK_ID_BUREAU",
    "previous_application.csv": "SK_ID_CURR",
    "POS_CASH_balance.csv": "SK_ID_PREV, SK_ID_CURR",
    "credit_card_balance.csv": "SK_ID_PREV, SK_ID_CURR",
    "installments_payments.csv": "SK_ID_PREV, SK_ID_CURR",
    "sample_submission.csv": "SK_ID_CURR",
    "HomeCredit_columns_description.csv": "-"
}

In [125]:
inventory_df["entity_type"] = inventory_df["file_name"].map(ENTITY_TYPES)
inventory_df["primary_key"] = inventory_df["file_name"].map(PRIMARY_KEYS)
inventory_df["foreign_keys"] = inventory_df["file_name"].map(FOREIGN_KEYS)

inventory_df = inventory_df[
    [
        "file_name",
        "entity_type",
        "rows",
        "columns",
        "size_mb",
        "primary_key",
        "foreign_keys",
        "candidate_keys"
    ]
]

inventory_df

,file_name,entity_type,rows,columns,size_mb,primary_key,foreign_keys,candidate_keys
0,application_test.csv,Main application - test,48744,121,25.34,SK_ID_CURR,-,SK_ID_CURR
1,application_train.csv,Main application - train,307511,122,158.44,SK_ID_CURR,-,SK_ID_CURR
2,bureau.csv,Credit bureau history,1716428,17,162.14,SK_ID_BUREAU,SK_ID_CURR,"SK_ID_CURR, SK_ID_BUREAU"
3,bureau_balance.csv,Credit bureau monthly status,27299925,3,358.19,"SK_ID_BUREAU, MONTHS_BALANCE",SK_ID_BUREAU,SK_ID_BUREAU
4,credit_card_balance.csv,Credit card monthly balance,3840312,23,404.91,"SK_ID_PREV, MONTHS_BALANCE","SK_ID_PREV, SK_ID_CURR","SK_ID_PREV, SK_ID_CURR"
5,HomeCredit_columns_description.csv,Data dictionary,219,5,0.04,-,-,-
6,installments_payments.csv,Installment payments history,13605401,8,689.62,"SK_ID_PREV, NUM_INSTALMENT_NUMBER, NUM_INSTALM...","SK_ID_PREV, SK_ID_CURR","SK_ID_PREV, SK_ID_CURR"
7,POS_CASH_balance.csv,POS cash monthly balance,10001358,8,374.51,"SK_ID_PREV, MONTHS_BALANCE","SK_ID_PREV, SK_ID_CURR","SK_ID_PREV, SK_ID_CURR"
8,previous_application.csv,Previous credit applications,1670214,37,386.21,SK_ID_PREV,SK_ID_CURR,"SK_ID_PREV, SK_ID_CURR"
9,sample_submission.csv,Kaggle submission template,48744,2,0.51,SK_ID_CURR,SK_ID_CURR,SK_ID_CURR


In [126]:
inventory_df[
    inventory_df[["entity_type", "primary_key", "foreign_keys"]]
    .isna()
    .any(axis=1)
]

,file_name,entity_type,rows,columns,size_mb,primary_key,foreign_keys,candidate_keys


In [127]:
# ==========================================================
# Export inventory to Markdown
# ==========================================================

inventory_output_path = DOCS_PATH / "inventario_dataset.md"

markdown_content = f"""# Inventario del Dataset

Este documento fue generado automáticamente desde el notebook:

`notebooks/01_eda_exploratorio/01_dataset_inventory.ipynb`

## Fuente de datos

Dataset: Home Credit Default Risk (Kaggle)

## Inventario de archivos

{inventory_df.to_markdown(index=False)}

## Observaciones iniciales

- application_train.csv es la tabla principal.
- application_test.csv contiene los datos para predicción.
- bureau contiene el historial crediticio externo.
- previous_application contiene solicitudes anteriores.
- credit_card_balance, POS_CASH_balance e installments_payments contienen información histórica de créditos.
- Las claves SK_ID_CURR, SK_ID_PREV y SK_ID_BUREAU serán la base del modelo relacional.
"""

inventory_output_path.write_text(markdown_content, encoding="utf-8")

print(f"Archivo generado en:\n{inventory_output_path}")

Archivo generado en:
c:\Users\Usuario\Desktop\RIESGO CREDITICIO\DATA-engineering\docs\inventario_dataset.md
